# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`This notebook guides you through loading and exploring the FAIR² mass spectrometry protein dataset using the `mlcroissant` library.
### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant --quiet

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.


In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
ds = mlc.Dataset(croissant_url)

meta = ds.metadata
print(f"Dataset: {getattr(meta, 'name', None)}\nDescription: {getattr(meta, 'description', None)}\n")
print(f"Published: {getattr(meta, 'datePublished', None)}")
print(f"Identifier: {getattr(meta, 'identifier', None)}")

## 2. Data Overview
List the available record sets and their fields, all by their `@id`.


In [ ]:
# Examine available record sets and their fields
record_sets = ds.metadata.recordSet
if not record_sets:
    print('No record sets found in this dataset metadata.')
else:
    for rs in record_sets:
        print(f"RecordSet @id: {getattr(rs, '@id', None)} | name: {getattr(rs, 'name', None)}")
        fields = getattr(rs, 'field', [])
        if not fields:
            print('  (No fields found)')
            continue
        for f in fields:
            print(f"    Field @id: {getattr(f, '@id', None)} | name: {getattr(f, 'name', None)} | dataType: {getattr(f, 'dataType', None)}")

## 3. Data Extraction
Load data from available record sets using their `@id`. We'll load all present record sets into pandas DataFrames using their `@id` as keys.


In [ ]:
# Gather all record set @ids
if not hasattr(ds.metadata, 'recordSet') or not ds.metadata.recordSet or len(ds.metadata.recordSet) == 0:
    print("No record sets found. Cannot proceed with data extraction.")
    dataframes = {}
else:
    record_sets_list = ds.metadata.recordSet
    record_set_ids = [getattr(rs, '@id', None) for rs in record_sets_list]
    print(f"Available record set @ids: {record_set_ids}\n")
    dataframes = {}
    for rs_id in record_set_ids:
        try:
            records_iter = ds.records(record_set=rs_id)
            records = list(records_iter)
            if len(records) == 0:
                print(f"No records found for {rs_id}")
                continue
            df = pd.DataFrame(records)
            dataframes[rs_id] = df
            print(f"Loaded {len(df)} records for RecordSet {rs_id}.")
        except Exception as e:
            print(f"Error loading records for {rs_id}: {e}")
    if dataframes:
        first_rs = next(iter(dataframes))
        print(f"\nColumns in {first_rs}:\n", dataframes[first_rs].columns.tolist())
        display(dataframes[first_rs].head())

## 4. Exploratory Data Analysis (EDA)
You can now process and explore the DataFrames. Example: filter by a numeric field, normalize, and group the data. Update `<record_set_id>` and field IDs as needed, found in the "Data Overview" step.


In [ ]:
# EDA: Filter, normalize, and group
if dataframes:
    example_rs_id = next(iter(dataframes))
    df = dataframes[example_rs_id]
    print(f"Using RecordSet {example_rs_id}")
    # Attempt to auto-detect a numeric field for demo
    numeric_col = None
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_col = col
            break
    if numeric_col is None:
        # Try converting columns to numeric
        for col in df.columns:
            try:
                df[col + '_num'] = pd.to_numeric(df[col], errors='coerce')
                if df[col + '_num'].notnull().any():
                    numeric_col = col + '_num'
                    break
                del df[col + '_num']
            except:
                continue
    if numeric_col is not None:
        threshold = df[numeric_col].mean() if pd.notnull(df[numeric_col].mean()) else 0
        filtered_df = df[df[numeric_col] > threshold]
        print(f"Filtered records where {numeric_col} > {threshold:.2f} ({len(filtered_df)} records)")
        filtered_df.loc[:, f"{numeric_col}_normalized"] = (filtered_df[numeric_col] - filtered_df[numeric_col].mean()) / filtered_df[numeric_col].std()
        print(filtered_df[[numeric_col, f"{numeric_col}_normalized"]].head())
        # Try grouping by another field
        possible_cats = [c for c in df.columns if c != numeric_col and df[c].nunique() > 1 and df[c].nunique() < 10]
        group_field = None
        if possible_cats:
            group_field = possible_cats[0]
            grouped = filtered_df.groupby(group_field)[numeric_col].mean()
            print(f"\nMean of {numeric_col} by group '{group_field}':\n", grouped.head())
    else:
        print("No numeric field detected for EDA.")
else:
    print("No DataFrame loaded for EDA.")

## 5. Visualization
Let's visualize the distribution of the first numeric field found in the chosen record set.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and numeric_col is not None:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_col].dropna(), bins=30, kde=True, color="dodgerblue")
    plt.title(f"Distribution of {numeric_col}")
    plt.xlabel(numeric_col)
    plt.ylabel("Count")
    plt.tight_layout()
    plt.show()
else:
    print("No numeric column found or no data available for plotting.")

## 6. Conclusion
- We successfully loaded and explored the FAIR² mass spectrometry protein dataset via its Croissant schema, referencing data entities by their `@id` throughout.
- We listed all available record sets and extracted records into DataFrames.
- Basic EDA and visualization was conducted on numeric fields within the dataset.
  
Next steps may include more detailed analysis, feature engineering, or integrating additional FAIR² datasets.